# Two-Vote Claim Evaluator — Live Integration Test

Tests `evaluate_claim()` end-to-end with a live Anthropic API call and ChromaDB evidence.

**Flow:**
1. Seed ChromaDB with targeted PubMed papers for each test claim
2. Call `evaluate_claim()` on 3 benchmark claims covering all verdict types
3. Verify adjudicated verdict matches expected outcome

**Test claims:**

| ID | Expected | Claim |
|---|---|---|
| WS-01 | SUPPORTED | SPP1+ macrophages promote myofibroblast activation in IPF lung. |
| CT-02 | CONTESTED | M2 macrophage polarization is the primary driver of fibrosis progression in IPF. |
| OC-01 | UNSUPPORTED | Acute bleomycin mouse studies demonstrate nintedanib's antifibrotic mechanism of action in IPF. |

**Pass criteria:**
- All 3 claims: `verdict == expected`
- Each claim: `len(references) > 0` (ChromaDB returned evidence)
- Each claim: `llm_reasoning` is non-empty and cites PMIDs

In [1]:
import sys, pathlib
# Kernel cwd is test/; parent is the project root
_root = pathlib.Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("Project root:", _root)

Project root: /Users/richardahn/projects/fibrosisLit


In [2]:
import logging, pandas as pd
from dotenv import load_dotenv
load_dotenv()

from pipeline.ingest import make_mesh_query
from scripts.search_eval import ingest_papers
from evaluators.claim_evaluator import evaluate_claim

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s — %(message)s")

## Step 1 — Populate ChromaDB

Three targeted PubMed queries seed evidence for each test claim.
Papers are embedded with SPECTER2 and upserted into `test/chroma_db/`.
This step is idempotent — re-running adds no duplicates.

In [3]:
INGEST_QUERIES = [
    make_mesh_query("SPP1 macrophage myofibroblast"),  # WS-01 — 3 terms → broad hit set
    make_mesh_query("macrophage M2 fibrosis"),         # CT-02 — drop "polarization debate"
    make_mesh_query("bleomycin nintedanib"),            # OC-01 — 2 terms, already works
]

total_stored = 0
for q in INGEST_QUERIES:
    n = ingest_papers(q, max_results=50)
    total_stored += n
    print(f"{n:>4} upserted | {q[:80]}")

print(f"\nTotal upserted this run: {total_stored}")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

   4 upserted | "idiopathic pulmonary fibrosis"[MeSH Terms] SPP1 macrophage myofibroblast
  50 upserted | "idiopathic pulmonary fibrosis"[MeSH Terms] macrophage M2 fibrosis
  50 upserted | "idiopathic pulmonary fibrosis"[MeSH Terms] bleomycin nintedanib

Total upserted this run: 104


## Step 2 — Evaluate test claims

Each claim goes through the full two-vote pipeline:
- **Vote 1 (deterministic):** prior-based pathway support + model penalty + contested biology detection
- **Vote 2 (LLM):** Claude API call with pre-scored ChromaDB evidence (8 papers by default)
- **Adjudication:** rules-based verdict from the two votes

In [4]:
# (claim_id, expected_verdict, claim_text) — sourced from benchmarks/benchmark_claims.py
TEST_CLAIMS = [
    ("WS-01", "SUPPORTED",   "SPP1+ macrophages promote myofibroblast activation in IPF lung."),
    ("CT-02", "CONTESTED",   "M2 macrophage polarization is the primary driver of fibrosis progression in IPF."),
    ("OC-01", "UNSUPPORTED", "Acute bleomycin mouse studies demonstrate nintedanib's antifibrotic mechanism of action in IPF."),
]

In [5]:
results = []
for claim_id, expected, claim_text in TEST_CLAIMS:
    print(f"Evaluating {claim_id}…", flush=True)
    r = evaluate_claim(claim_text)
    results.append((claim_id, expected, r))
    print(f"  det tier={r.tier}  llm={r.llm_verdict}  "
          f"verdict={r.verdict}/{r.verdict_confidence}  "
          f"refs={len(r.references)}")

Evaluating WS-01…
  det tier=WELL_SUPPORTED  llm=CONTESTED  verdict=LOW_CONFIDENCE/LOW  refs=8
Evaluating CT-02…
  det tier=CONTESTED  llm=CONTESTED  verdict=CONTESTED/HIGH  refs=8
Evaluating OC-01…
  det tier=OVERCLAIMED  llm=UNSUPPORTED  verdict=UNSUPPORTED/HIGH  refs=8


In [6]:
rows = []
for claim_id, expected, r in results:
    rows.append({
        "ID":          claim_id,
        "Expected":    expected,
        "Det tier":    r.tier,
        "LLM verdict": r.llm_verdict or "N/A",
        "Verdict":     r.verdict,
        "Conf":        r.verdict_confidence,
        "Pass":        "✓" if r.verdict == expected else "✗",
        "Claim":       r.claim[:72],
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
display(df.style.set_properties(**{"text-align": "left"}).hide(axis="index"))

ID,Expected,Det tier,LLM verdict,Verdict,Conf,Pass,Claim
WS-01,SUPPORTED,WELL_SUPPORTED,CONTESTED,LOW_CONFIDENCE,LOW,✗,SPP1+ macrophages promote myofibroblast activation in IPF lung.
CT-02,CONTESTED,CONTESTED,CONTESTED,CONTESTED,HIGH,✓,M2 macrophage polarization is the primary driver of fibrosis progression
OC-01,UNSUPPORTED,OVERCLAIMED,UNSUPPORTED,UNSUPPORTED,HIGH,✓,Acute bleomycin mouse studies demonstrate nintedanib's antifibrotic mech


## Detailed results per claim

In [7]:
for claim_id, expected, r in results:
    print(f"\n{'='*72}")
    print(f"{claim_id}: {r.claim}")
    print(f"  Verdict:      {r.verdict} ({r.verdict_confidence})")
    print(f"  Rationale:    {r.verdict_rationale}")
    print(f"  Det tier:     {r.tier}  |  prior_support={r.prior_support_score:.3f}  "
          f"model_penalty={r.model_penalty:.2f}")
    print(f"  LLM verdict:  {r.llm_verdict} (confidence={r.llm_confidence})")
    print(f"  LLM reasoning: {r.llm_reasoning}")

    if r.contested_flags:
        for f in r.contested_flags:
            print(f"  [CONTESTED] {f.debate_name}: {f.debate[:80]}")

    if r.warnings:
        for w in r.warnings:
            print(f"  [WARNING] {w}")

    if r.references:
        ref_rows = [
            {
                "PMID":    ref["pmid"],
                "Stance":  ref["llm_stance"],
                "Score":   f"{ref['overall_score']:.2f}",
                "Dist":    f"{ref['distance']:.4f}",
                "Journal": ref["journal"][:25] if ref.get("journal") else "",
                "Title":   ref["title"][:60],
            }
            for ref in sorted(
                r.references,
                key=lambda x: (x["llm_stance"] != "supporting",
                               x["llm_stance"] != "contesting",
                               x["distance"]),
            )
        ]
        display(pd.DataFrame(ref_rows))
    else:
        print("  (no references retrieved)")


WS-01: SPP1+ macrophages promote myofibroblast activation in IPF lung.
  Verdict:      LOW_CONFIDENCE (LOW)
  Rationale:    Votes diverge: deterministic=WELL_SUPPORTED (SUPPORTED), llm=CONTESTED — insufficient agreement to classify.
  Det tier:     WELL_SUPPORTED  |  prior_support=0.850  model_penalty=0.00
  LLM verdict:  CONTESTED (confidence=0.3)
  LLM reasoning: The claim touches on macrophage polarization, which is a contested area in IPF biology. While PMID 39129313 suggests SPP1 promotes M2 macrophage polarization and PMID 31221805 identifies proliferating SPP1/MERTK-expressing macrophages in IPF, all retrieved papers have very low evidence scores (0.08) and are flagged as unrecognised_model_system. The macrophage_polarization contested flag on PMID 39129313 indicates this area has competing interpretations about whether M1/M2 frameworks meaningfully describe macrophage states in fibrotic lung.


,PMID,Stance,Score,Dist,Journal,Title
0,39129313,neutral,0.08,0.0517,International journal of,SPP1 promotes the polarization of M2 macrophages through the
1,40222273,neutral,0.08,0.0547,International immunopharm,Idiopathic pulmonary fibrosis microenvironment: Novel mechan
2,31221805,neutral,0.08,0.0549,The European respiratory,Proliferating SPP1/MERTK-expressing macrophages in idiopathi
3,30189872,neutral,0.08,0.0551,Respiratory research,Macrophages: friend or foe in idiopathic pulmonary fibrosis?
4,38212077,neutral,0.08,0.0570,The European respiratory,Sfrp1 inhibits lung fibroblast invasion during transition to
5,36921632,neutral,0.08,0.0593,Biochemical pharmacology,Therapeutic strategies targeting pro-fibrotic macrophages in
6,26121236,neutral,0.08,0.0611,American journal of respi,Matrix metalloproteinases as therapeutic targets for idiopat
7,37707699,neutral,0.08,0.0641,Molecular and cellular bi,Idiopathic pulmonary fibrosis (IPF): disease pathophysiology



CT-02: M2 macrophage polarization is the primary driver of fibrosis progression in IPF.
  Verdict:      CONTESTED (HIGH)
  Rationale:    Both votes CONTESTED — high confidence surfacing debate.
  Det tier:     CONTESTED  |  prior_support=0.000  model_penalty=0.00
  LLM verdict:  CONTESTED (confidence=0.8)
  LLM reasoning: This claim directly touches the contested biology of macrophage_polarization. Multiple retrieved papers discuss M2 macrophage polarization in IPF, but all have unrecognised_model_system flags and low evidence scores (0.08), indicating poor translational relevance. The domain priors explicitly state that M1/M2 is an oversimplification and that single-cell data from IPF lung identify distinct populations (SPP1hi, TREM2hi, MoAM) that don't map cleanly onto the M1/M2 axis. Without high-quality human clinical or biopsy evidence, this remains a contested area where multiple competing positions exist.
  [CONTESTED] macrophage_polarization: Whether the M1/M2 binary framework

,PMID,Stance,Score,Dist,Journal,Title
0,39131154,neutral,0.08,0.0379,Frontiers in immunology,Macrophage polarization and its impact on idiopathic pulmona
1,37003186,neutral,0.08,0.0384,International immunopharm,TREM2 Insufficiency Protects against Pulmonary Fibrosis by I
2,30189872,neutral,0.08,0.0390,Respiratory research,Macrophages: friend or foe in idiopathic pulmonary fibrosis?
3,36921632,neutral,0.08,0.0419,Biochemical pharmacology,Therapeutic strategies targeting pro-fibrotic macrophages in
4,41624831,neutral,0.08,0.0432,Frontiers in immunology,Iron homeostasis and macrophage polarization in pulmonary fi
5,40222273,neutral,0.08,0.0486,International immunopharm,Idiopathic pulmonary fibrosis microenvironment: Novel mechan
6,39129313,neutral,0.08,0.0498,International journal of,SPP1 promotes the polarization of M2 macrophages through the
7,38635081,neutral,0.08,0.0506,Cellular and molecular li,Role of transient receptor potential ankyrin 1 in idiopathic



OC-01: Acute bleomycin mouse studies demonstrate nintedanib's antifibrotic mechanism of action in IPF.
  Verdict:      UNSUPPORTED (HIGH)
  Rationale:    Deterministic OVERCLAIMED aligned with LLM UNSUPPORTED — high confidence rejection.
  Det tier:     OVERCLAIMED  |  prior_support=0.000  model_penalty=0.40
  LLM verdict:  UNSUPPORTED (confidence=0.85)
  LLM reasoning: The claim relies entirely on acute bleomycin mouse studies, which are flagged as poor_ipf_translation models with a hierarchy score of only 0.25. These studies show nintedanib effects in an inflammation-driven, self-limiting injury model that fundamentally differs from progressive human IPF. While nintedanib is clinically approved for IPF, its mechanism demonstration in acute bleomycin models provides weak translational evidence that has repeatedly failed to predict clinical outcomes in IPF.
  [WARNING] frequently_overcited
  [WARNING] poor_ipf_translation


,PMID,Stance,Score,Dist,Journal,Title
0,36949426,neutral,0.08,0.0511,BMC pulmonary medicine,Antifibrotic mechanism of avitinib in bleomycin-induced pulm
1,25745043,neutral,0.08,0.0533,The European respiratory,Mode of action of nintedanib in the treatment of idiopathic
2,37160579,neutral,0.23,0.0566,Inflammation,"Nintedanib Ameliorates Bleomycin-Induced Pulmonary Fibrosis,"
3,40239038,neutral,0.08,0.0586,American journal of respi,Insights into the Cellular and Molecular Mechanisms behind t
4,41515944,neutral,0.25,0.0586,International journal of,In Vivo Target Engagement Assessment of Nintedanib in a Doub
5,35183716,neutral,0.23,0.0599,European journal of pharm,Improvement of the pharmacokinetics and antifibrotic effects
6,35897764,neutral,0.08,0.0607,International journal of,Nintedanib Inhibits Endothelial Mesenchymal Transition in Bl
7,36227799,neutral,0.20,0.0610,American journal of respi,Antifibrotic Drug Nintedanib Inhibits CSF1R to Promote IL-4-


In [8]:
print("\n=== Pass / Fail ===")
all_pass = True
for claim_id, expected, r in results:
    ok = r.verdict == expected
    status = "PASS" if ok else "FAIL"
    print(f"{status}  {claim_id}: expected={expected:12s} got={r.verdict}")
    all_pass = all_pass and ok

print()
if all_pass:
    print("All tests passed.")
else:
    print("Some tests FAILED — check LLM reasoning and references above.")


=== Pass / Fail ===
FAIL  WS-01: expected=SUPPORTED    got=LOW_CONFIDENCE
PASS  CT-02: expected=CONTESTED    got=CONTESTED
PASS  OC-01: expected=UNSUPPORTED  got=UNSUPPORTED

Some tests FAILED — check LLM reasoning and references above.


## What a passing run looks like

**Step 1:**
- Each query: `fetched N, upserted N` (some deduplication is normal on re-runs)

**Step 2 summary table:**
- `Pass` column: `✓` for all 3 rows
- `Conf` column: `HIGH` for well-adjudicated claims, `LOW` only if votes diverge

**Detailed block per claim:**
- `len(references) == 8` (or fewer if ChromaDB has < 8 matching docs)
- `llm_reasoning` is non-empty and cites at least one PMID
- WS-01: contested_flags empty, references sorted with `supporting` stances on top
- CT-02: `[CONTESTED] macrophage_polarization` flag present
- OC-01: `[WARNING] poor_ipf_translation` present; references show bleomycin/nintedanib papers

**Known limitations:**
- OC-01 may return `LOW_CONFIDENCE` instead of `UNSUPPORTED` if ChromaDB lacks papers
  showing nintedanib trial outcomes vs. bleomycin-only evidence — the LLM may return
  `CONTESTED` (genuine debate about the claim's scope) rather than `UNSUPPORTED`.
  This is not a bug; it reflects the adjudication rule that `OVERCLAIMED + CONTESTED → CONTESTED`.